[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/internshipinabook/cv-internship-in-a-book/blob/main/notebooks/week8_uterine_cancer_STARTER.ipynb)


# Week 8 -- Uterine Cancer — Pathology at Scale (STARTER)
### The Computer Vision Internship | MediVision AI Lagos

**This week:** TCGA uterine cancer — WSI patches, stain normalisation (Macenko), EfficientNet on pathology

**Dataset:** TCGA Uterine cancer patches

**STARTER notebook** -- your working environment. Complete each exercise before moving on.


In [ ]:
# -- Colab/Local Setup -- run this first in Colab, skip locally -------------
import os

if not os.path.exists('requirements.txt'):
    # Clone the full course repository
    !git clone https://github.com/internshipinabook/cv-internship-in-a-book.git cv-book
    %cd cv-book
    # Install all required packages (suppress verbose output with -q)
    !pip install -r requirements.txt -q

# Create working directories
os.makedirs('outputs',  exist_ok=True)   # charts, reports, predictions
os.makedirs('models',   exist_ok=True)   # trained model checkpoints
os.makedirs('datasets', exist_ok=True)   # raw downloaded data
print('Setup complete.')


In [ ]:
# -- Reproducibility Seeds --------------------------------------------------
# Setting seeds ensures every random operation produces the same result
# on every run. Essential for: comparing runs, debugging, sharing results.

import random    # Python built-in random
import numpy as np   # NumPy random (data loading, sklearn)
import torch     # PyTorch random (neural networks, dropout)

SEED = 42        # 42 is conventional -- any integer works

random.seed(SEED)           # fix Python random() calls
np.random.seed(SEED)        # fix np.random.* calls
torch.manual_seed(SEED)     # fix PyTorch CPU operations

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)  # fix GPU operations too

# Prefer deterministic cuDNN algorithms (may slow training ~5%)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Seeds fixed: {SEED} | Device: {device}')


## Learning Objectives

By the end of this week, you will be able to:

- Load TCGA uterine endometrial cancer patches and describe what makes them clinically distinct from BreakHis
- Apply Macenko stain normalisation and measure its effect on inter-site variability
- Fine-tune EfficientNet-B0 for binary cancer vs normal classification
- Evaluate the model per TCGA contributing site and identify the site gap
- Write a model card for the uterine cancer classifier



---

## MONDAY -- "Endometrial Cancer — The Clinical Context"


Endometrial cancer arises from the lining of the uterus (endometrium). The most common histological type is endometrioid adenocarcinoma (Type I, oestrogen-driven, better prognosis). Type II includes serous carcinoma and clear cell carcinoma — more aggressive, less common, harder to classify visually. The TCGA uterine dataset contains both types, creating a within-cancer-type classification challenge.


### Exercise 8.1 -- Clinical context

Research: what is the 5-year survival rate for Type I vs Type II endometrial cancer in Nigeria? What is the average time from first symptom to diagnosis? Write a one-paragraph clinical brief for a non-specialist.


In [ ]:
# Exercise 8.1: Clinical context
# -------------------------------------------------------
# YOUR CODE HERE
# Hint: read the markdown cell above carefully.
# Hint: check the Common Mistakes section at the bottom of this notebook.


**Monday code scaffold** (read and run, then adapt in the exercise cells above):


In [ ]:
# -- Monday: Endometrial Cancer — The Clinical Context (scaffold) --
# TCGA Uterine dataset structure
# tissue/
#   endometrioid/    # Type I — most common
#     TCGA-2E-A9G8/  # patient case ID
#       patches/     # 256×256 WSI patches at 20× magnification
#   serous/          # Type II — more aggressive
#     TCGA-A5-A0GD/
#       patches/
#   normal/          # Normal endometrium for comparison
#     TCGA-B4-5377/

# Clinical task: binary classification
# Class 0: Normal endometrium
# Class 1: Cancer (endometrioid OR serous)


### Monday Morning Moment

*Slack — Monday, 9:30am.*

**Ngozi Eze-Okafor:** The TCGA patches look different from BreakHis. More variation in colour intensity between slides.

**Dr. Kwame Asante:** That is the between-institution variability. How many contributing sites does the TCGA uterine dataset have?

**Ngozi Eze-Okafor:** I can check — the metadata file should have that. Let me see... 26 sites.

**Dr. Kwame Asante:** 26 labs, 26 staining protocols, 26 microscope models. What is the consequence for a model trained on a random 80% split?

**Ngozi Eze-Okafor:** The train and test sets will contain patches from the same sites. So the model learns site-specific staining as a feature. When it encounters a new site it has never seen —

**Dr. Kwame Asante:** It fails. Which is why site-level splitting matters as much as slide-level splitting. Write that in your preprocessing log.




---

## TUESDAY -- "Stain Normalisation — Macenko in Practice"


TCGA patches come from 25+ contributing institutions. Each institution uses slightly different haematoxylin-eosin concentrations. A model trained on patches from MD Anderson will fail on patches from Mayo Clinic — not because the tumours differ, but because the purple is a different shade. Macenko normalisation fits a stain deconvolution model on a reference patch and normalises all other patches to match.


### Exercise 8.2 -- Stain normalisation effect

Apply Macenko normalisation to patches from 3 different TCGA sites. Compute: per-channel mean and std before and after, per-site variance before and after. Does normalisation reduce between-site variance?


In [ ]:
# Exercise 8.2: Stain normalisation effect
# -------------------------------------------------------
# YOUR CODE HERE
# Hint: read the markdown cell above carefully.
# Hint: check the Common Mistakes section at the bottom of this notebook.


**Tuesday code scaffold** (read and run, then adapt in the exercise cells above):


In [ ]:
# -- Tuesday: Stain Normalisation — Macenko in Practice (scaffold) --
from torchstain import MacenkoNormalizer
import torch
from torchvision.transforms.functional import to_tensor, to_pil_image
from PIL import Image

# Load a well-stained reference patch from one institution
ref = to_tensor(Image.open("datasets/tcga/reference_patch.png").convert("RGB")).unsqueeze(0)
normalizer = MacenkoNormalizer()
normalizer.fit(ref)

# Normalise a patch from a different institution
target = to_tensor(Image.open("datasets/tcga/patch_mayo.png").convert("RGB")).unsqueeze(0)
norm_patch = normalizer.normalize(target)

# Quantify the effect: compare per-channel std before and after
print(f"Before: std = {target.std():.4f}")
print(f"After:  std = {norm_patch.std():.4f}")



---

## WEDNESDAY -- "EfficientNet-B0 — Compound Scaling for Pathology"


EfficientNet scales width, depth, and resolution together rather than independently. EfficientNet-B0 (5.3M parameters) outperforms ResNet-50 (25M parameters) on many tasks while being 5× smaller. For pathology: smaller models run on the CPU workstations common in Nigerian hospitals. B0 is the right choice here — not because it is best in absolute terms, but because it is the best model that a non-GPU clinical machine can run.


### Exercise 8.3 -- EfficientNet variants

Compare EfficientNet-B0, B1, and B2 on the same held-out test set. Report: AUC, F1, parameter count, inference time on CPU. Which offers the best AUC-per-parameter trade-off?


In [ ]:
# Exercise 8.3: EfficientNet variants
# -------------------------------------------------------
# YOUR CODE HERE
# Hint: read the markdown cell above carefully.
# Hint: check the Common Mistakes section at the bottom of this notebook.


**Wednesday code scaffold** (read and run, then adapt in the exercise cells above):


In [ ]:
# -- Wednesday: EfficientNet-B0 — Compound Scaling for Pathology (scaffold) --
import timm

# EfficientNet-B0 pretrained on ImageNet
model = timm.create_model("efficientnet_b0", pretrained=True, num_classes=2)

# EfficientNet uses its own normalisation convention
from timm.data import resolve_data_config, create_transform
config    = resolve_data_config({}, model=model)
transform = create_transform(**config)

# Check: how many parameters?
n_params = sum(p.numel() for p in model.parameters())
print(f"EfficientNet-B0 parameters: {n_params/1e6:.1f}M")
print(f"ResNet-18 parameters:       11.7M  (for comparison)")



---

## THURSDAY -- "Per-Site Evaluation — The Site Gap"


TCGA records which institution contributed each case. Evaluate model performance separately for the five largest contributing sites. The site gap (max AUC - min AUC across sites) is the fairness metric for this model. A site gap > 0.10 AUC suggests the model performs systematically differently across institutions — a safety concern for deployment.


### Exercise 8.4 -- Site-level splitting

Implement site-level train/test split (no site in both sets). Compare AUC to image-level random split. Quantify the data leakage effect: how much did random splitting inflate AUC?


In [ ]:
# Exercise 8.4: Site-level splitting
# -------------------------------------------------------
# YOUR CODE HERE
# Hint: read the markdown cell above carefully.
# Hint: check the Common Mistakes section at the bottom of this notebook.


**Thursday code scaffold** (read and run, then adapt in the exercise cells above):


In [ ]:
# -- Thursday: Per-Site Evaluation — The Site Gap (scaffold) --
# Per-site evaluation
site_col = "contributing_site"  # column in metadata CSV
results  = {}
for site in df[site_col].unique():
    site_mask = df[site_col] == site
    if site_mask.sum() < 20: continue  # skip small sites
    site_preds = predictions[site_mask]
    site_true  = labels[site_mask]
    auc = roc_auc_score(site_true, site_preds)
    results[site] = {"auc": auc, "n": site_mask.sum()}

site_gap = max(r["auc"] for r in results.values()) - \
           min(r["auc"] for r in results.values())
print(f"Site gap (max-min AUC): {site_gap:.4f}")
print(f"Clinical threshold: 0.10 (>{0.10} = safety concern)")



---

## FRIDAY -- "The Friday Build: The Model Card"


Write the complete model card for the uterine cancer classifier. Sections: model details, intended use, out-of-scope use, training data, evaluation results (per-site AUC), fairness findings (site gap), limitations, and deployment recommendation. The limitations section must be written before the performance section — it keeps you honest.


**Friday code scaffold** (read and run, then adapt in the exercise cells above):


In [ ]:
# -- Friday: The Friday Build: The Model Card (scaffold) --
# Model card must answer:
# 1. What does this model do and for whom?
# 2. What data was it trained on?
# 3. How does it perform — per site, not just aggregate?
# 4. Who is it likely to fail for?
# 5. Should it be deployed? Under what conditions?
# Mitchell et al. (2019) model card framework — see references


### Friday Workplace Moment

*MediVision AI — Friday standup.*

**Ngozi Eze-Okafor:** Site gap before stain normalisation: 0.19 AUC. After Macenko normalisation: 0.11. Still above the 0.10 safety threshold, but reduced by 42%.

**Dr. Kwame Asante:** What accounts for the remaining 0.11 gap after normalisation?

**Ngozi Eze-Okafor:** Three hypotheses: different tissue preparation (not just staining), different scanner resolution per site, and case-mix differences — some sites contribute more aggressive serous carcinoma.

**Nurse Folake Balogun:** If we deploy this at Murtala Mohammed Specialist Hospital in Kano, is it safe?

**Ngozi Eze-Okafor:** Not until we evaluate it on slides from Kano specifically. We would need at least 50 labelled cases from that site before I could answer that question.

**Dr. Kwame Asante:** That goes in the model card under "Deployment Requirements".



## YOUR WEEK 8 CHECKLIST

Check each item before moving to the next week.
If you cannot explain an item without notes, revisit that section.

- [ ] I can explain what endometrial cancer is and distinguish Type I from Type II clinically.
- [ ] I applied Macenko stain normalisation and measured its effect on the site gap in AUC.
- [ ] I fine-tuned EfficientNet-B0 and can explain why B0 was chosen over larger variants for this deployment context.
- [ ] I evaluated per contributing site and can state the site gap before and after normalisation from memory.
- [ ] My model card has a Limitations section written before the Performance section.


## Challenge Task

> **Core path:** Implement site-level cross-validation (leave-one-site-out). Which site produces the largest drop in AUC when used as the test set? What property of that site explains the drop?

> **Advanced path:** Implement the Vahadane stain normalisation method and compare to Macenko on the site gap metric. Which method produces smaller post-normalisation site variance?


## Common Mistakes

**Splitting TCGA data randomly across images or cases:** Multiple patches from the same slide, and slides from the same site, create data leakage. Split by case ID first, then evaluate per site.

**Using EfficientNet's default ImageNet normalisation on pathology images:** timm.create_model returns a model with ImageNet normalisation. Use resolve_data_config to get the model's expected normalisation parameters.

**Reporting only aggregate AUC:** Aggregate AUC of 0.89 hides a site gap of 0.19. Per-site reporting is required for clinical deployment decisions.



---

## Get the Full Book

This notebook is part of **The Computer Vision Internship** -- Book 3 of 9, InternshipInABook(tm).

Covers: natural images, medical X-rays, breast & uterine cancer, video, GradCAM/LIME/SHAP/Integrated Gradients, and fairness audits.

Available at [internshipinabook.com](https://internshipinabook.com)
